In [1]:
print(1+1)

2


In [2]:
%%capture
!pip install --force-reinstall --no-cache-dir tenacity==8.2.3 --user
!pip install "ibm-watsonx-ai==1.0.8" --user
!pip install "ibm-watson-machine-learning==1.0.367" --user
!pip install "langchain-ibm==0.1.7" --user
!pip install "langchain-community==0.2.10" --user
!pip install "langchain-experimental==0.0.62" --user
!pip install "langchainhub==0.1.18" --user
!pip install "langchain==0.2.11" --user
!pip install "pypdf==4.2.0" --user
!pip install "chromadb==0.4.24" --user

In [ ]:
import os
os._exit(00)

: 

In [2]:
# You can also use this section to suppress warnings generated by your code:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')
import os
os.environ['ANONYMIZED_TELEMETRY'] = 'False'

from ibm_watsonx_ai.foundation_models import ModelInference
from ibm_watsonx_ai.metanames import GenTextParamsMetaNames as GenParams
from ibm_watsonx_ai.foundation_models.utils.enums import ModelTypes
from ibm_watson_machine_learning.foundation_models.extensions.langchain import WatsonxLLM


In [3]:
model_id = 'meta-llama/llama-3-3-70b-instruct' 

parameters = {
    
    GenParams.MAX_NEW_TOKENS : 256,
    GenParams.TEMPERATURE : 0.2,
}

credentials = {
    "url" : "https://us-south.ml.cloud.ibm.com"
}


project_id = "skills-network"
model = ModelInference(
    
    model_id = model_id,
    params = parameters,
    credentials = credentials ,
    project_id = project_id
)



'`api_key` for IAM token is not provided in credentials for the client'


WMLClientError: '`api_key` for IAM token is not provided in credentials for the client'

In [4]:
msg = model.generate("In todays sales meeting, we ")
print(msg['results'][0]['generate_text'])

NameError: name 'model' is not defined

In [5]:
llama_llm = WatsonxLLM(model = model)

NameError: name 'model' is not defined

In [6]:
print(llama_llm.invoke("Who is mans best friend?"))

NameError: name 'llama_llm' is not defined

In [7]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

In [ ]:
msg = llama_llm.invoke(
    [
        SystemMessage(content="You are a supportive AI bot that suggests fitness activities to a user in one short sentence"),
        HumanMessage(content="I like high-intensity workouts, what should I do?"),
        AIMessage(content="You should try a CrossFit class"),
        HumanMessage(content="How often should I attend?")
    ]
)

NameError: name 'llama_llm' is not defined

In [9]:
#AIMESSAGE = request to invoke tool to ai by messaging to AI
#HumanMessage = Request Prompt by user
#SystemMessage = Prime AI behaviour passed as prompt

In [10]:
##EXCERCISE 1

In [ ]:
# Define different parameter sets
parameters_creative = {
    GenParams.MAX_NEW_TOKENS: 256,
    GenParams.TEMPERATURE: 0.5,  # Higher temperature for more creative responses
}

parameters_precise = {
    GenParams.MAX_NEW_TOKENS: 256,
    GenParams.TEMPERATURE: 0.1,  # Lower temperature for more deterministic responses
}

# Define the model ID for ibm/granite-3-3-8b-instruct
granite='ibm/granite-3-8b-instruct'

# Define the model ID for llama-4-maverick-17b-128e-instruct-fp8
llama='meta-llama/llama-3-2-90b-vision-instruct'

credentials = {
    "url": "https://us-south.ml.cloud.ibm.com"
    # "api_key": "your api key here"
    # uncomment above and fill in the API key when running locally
}

model_llama = ModelInference(
    model_id = llama,
    credentials = credentials,
    project_id= project_id,
    params = parameters_creative
)

granite_model = ModelInference(
    model_id = granite,
    project_id = project_id,
    credentials= credentials,
    params = parameters_precise
)

llama_llm = WatsonxLLM(model = model_llama)
granite_llm = WatsonxLLM(model = granite_model)
# TODO: Send identical prompts to both models and comapre the responses.

poet_msg = llama_llm.invoke(

    [
      SystemMessage(content= "Imagine that you are a AIKillMachine"),
        HumanMessage(content = "Would you destroy humans as they are less optimised to adapt as compared to AI")
    ]
)

fact_msg = llama_llm.invoke(

    [
        SystemMessage(content = "Imagine you are an expert in Sciences"),
        HumanMessage(content = "What are the key components of a neural network?"),
        AIMessage(content = "Use Wikipedia and Britannica")
        
    ]
)

instruction_msg = granite_llm.invoke(
    [
        SystemMessage(content= "You are an AIBot designed to help"),
        HumanMessage(content= "List 5 tips for effective time management. Do it step by step breakdown")        
        
    ]
)

In [13]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [16]:
prompt = PromptTemplate.from_template("Tell me {adj} joke about {topic}")
input = {"adj" : "funny", "topic": "cats"}

In [17]:
prompt.invoke(input)

StringPromptValue(text='Tell me funny joke about cats')

In [19]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    
    ("system", "You are a helpful assistant"),
    ("user", "Tell me ajoke about {topic}")
    
])

input = {"topic" : "cats"} ## List of templates

prompt.invoke(input)

ChatPromptValue(messages=[SystemMessage(content='You are a helpful assistant'), HumanMessage(content='Tell me ajoke about cats')])

In [22]:
from langchain_core.prompts import MessagesPlaceholder

from langchain_core.messages import HumanMessage

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant"),
        MessagesPlaceholder("msgs")
    ]
)

input = {"msgs": [HumanMessage(content="What is the day after Tuesday?")]}
prompt.invoke(input)

ChatPromptValue(messages=[SystemMessage(content='You are a helpful assistant'), HumanMessage(content='What is the day after Tuesday?')])

In [ ]:
chain = prompt | granite_llm
response = chain.invoke(input = input)
print(response)

In [ ]:
## JSON vs CSV output parsing

In [23]:
from langchain_core.output_parsers import StrOutputParser

from langchain_core.pydantic_v1 import BaseModel, Field



In [24]:
class Joke(BaseModel):
    setup : str = Field(description="question to set up a joke")
    punchline : str = Field(description= "answer to resolve the joke")

In [ ]:
joke_query = "Tell me a joke"
output_parser = JsonOutputParser(pydantic_object = Joke)

format_instructions = output_parser.get_format_instructions()

prompt = PromptTemplate(
    template = "Answer the user query. \n{format_instructions}\n{query}\n",
    input_variables = ["query"],
    partial_variables = {"format_instructions" : format_instructions},
)

chain = prompt | llama_llm | output_parser
chain.invoke({"query": joke_query})

In [ ]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser

output_parser = CommaSeparatedListOutputParser()

format_instructions = output_parser.get_format_instructions()

prompt = PromptTemplate(
    template = "Answer the user query.{format_instructions}\n List  five {subject}",
    input_variables = ["subject"],
    partial_variables = {"format_instructions" : format_instructions},
)

chain = prompt | llama_llm | output_parser

chain.invoke({"subject" : "ice cream flavors"})